[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sheik-info/gnn_project/blob/master/predict_link.ipynb)

**sujet du projet dnn:**


Prédiction de liens : En cachant certains liens, peut-on parvenir à les redécouvrir? Ce problème peut être évalué classiquement en utilisant l'AUC, Average Precision, Hits@k et MRR.

In [1]:
!pip install torch_geometric
import matplotlib.pyplot as plt
import networkx as nx
import seaborn as sns
import torch as th
import torch.nn.functional as F
from matplotlib.cm import ScalarMappable
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import LabelEncoder
from torch import nn
from torch_geometric.nn import VGAE, GCNConv
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.utils import from_networkx, to_networkx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 46.1 MB/s eta 0:00:00


In [ ]:
G = nx.read_graphml("airportsAndCoordAndPop.graphml")
nx.draw(G, node_size=15)

In [29]:
data = from_networkx(G, group_node_attrs=["population", "lat", "lon"])


In [35]:
print(data)
print(data.keys())
print(data['edge_index'].shape)

Data(edge_index=[2, 27094], country=[3363], city_name=[3363], x=[3363, 3])
['city_name', 'x', 'country', 'edge_index']
torch.Size([2, 27094])


In [37]:
#on va essayer de deviner les noeuds des pays qui ont été supprimés
#on encode ce qu'on veut avoir en sortie y, ce qu'on veut deviner
encoder1 = LabelEncoder()
integer_labels = encoder1.fit_transform(data.country)
target_tensor1 = th.tensor(integer_labels, dtype=th.long)
print("target_tensor: ", target_tensor1)
data.y = target_tensor1
data.num_classes = len(set(data.country))

target_tensor:  tensor([ 66,  66,  66,  ...,  33, 143, 143])


In [ ]:
# encoder2 = LabelEncoder()
# integer_labels = encoder2.fit_transform(data.city_name)
# target_tensor2 = th.tensor(integer_labels, dtype=th.long)
# print("target_tensor: ", target_tensor2)
# data.y = target_tensor2
# data.num_classes = len(set(data.city_name))

In [38]:
num_nodes = data.num_nodes
train_ratio = 0.70

# Randomly creating a mask
#va créer un dataset avec 70 % d'avoir true et 30% d'avoir false
mask = th.rand(num_nodes) < train_ratio
print("masks: ", mask.shape, mask.sum())
data.train_mask = mask
#inverse les true et les false
data.test_mask = ~data.train_mask

# remove the attributes for the nodes that are not in the training set
#mettre des 0 dans x pour ne pas avoir les préférences et pouvoir les prédire
temp = th.zeros((num_nodes, 3), dtype=th.float)
temp[data.train_mask] = data.x[data.train_mask]
data.x = temp
data
print(data.y)

masks:  torch.Size([3363]) tensor(2326)
tensor([ 66,  66,  66,  ...,  33, 143, 143])


On va utiliser le VGAE pour faire de la prédictiion de liens

In [ ]:
#est non dirigé donc on a pas besoin de val_vgae mais on est obligé de le garder
#linksplit indispensable pour la prédiction de liens
transform = RandomLinkSplit(is_undirected=True)
train_vgae, val_vgae, test_vgae = transform(data)